
# NFL Big Data Bowl 2026 - Prediction: spatio-temporal trajectory model (The solution of the 5th place in the competition).

Self-contained reference implementation of a delta-prediction architecture
inspired by public leaderboard solutions (see REFERENCES at the bottom of this
file). Pipeline:

  input frames (pre-pass)  ->  temporal encoder (SqueezeFormer over time)
  
                             -> spatial encoder (Transformer over players,
                                distance-matrix attention bias)
                                
                             -> per-(player, horizon) tokens with RoPE
                             
                             -> BiLSTM || BiGRU over horizons
                             
                             -> spatio-temporal refinement (Transformer)
                             
                             -> head -> (dx, dy) per player per future frame

**Training**: Huber loss on per-frame deltas, Muon for 2D weights + AdamW for the
rest, EMA(0.995) weights for evaluation, cosine schedule with warmup.
Evaluation: official RMSE on absolute coordinates (deltas are cumsum-ed),
block-bootstrap confidence intervals, per-role / per-horizon breakdowns.

**Reproducibility**: single config with fixed seeds, deterministic dataloaders,
official week-based split, one-command scripts (see README snippet in main()).

**REFERENCES (mandatory attribution for the course report)**

  [1] NFL Big Data Bowl 2026 - Prediction, Kaggle competition & data.
  
  [2] Public 5th-place solution summary (architecture diagram: delta targets,
      RoPE over output frames, SqueezeFormer/Transformer spatio-temporal
      blocks, BiLSTM+BiGRU, Huber loss, Muon optimizer, EMA 0.995).
      
  [3] Muon optimizer: https://github.com/KellerJordan/Muon
  
  [4] RoPE: Su et al., "RoFormer: Enhanced Transformer with Rotary Position
      Embedding", 2021.
      
  [5] SqueezeFormer: Kim et al., 2022.

## 1. Environment, Seeds, Config., Data Search

In [ ]:
import os, math, json, random, pickle, time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import LambdaLR
from tqdm.notebook import tqdm

WORK = Path("/kaggle/working")
CKPT_ROOT, OUT_ROOT, FIG_ROOT = (WORK / d for d in ("checkpoints", "outputs", "figures"))
for d in (CKPT_ROOT, OUT_ROOT, FIG_ROOT):
    d.mkdir(parents=True, exist_ok=True)
CACHE_PATH = WORK / "prep_cache.pkl"


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed); os.environ["PYTHONHASHSEED"] = str(seed)


def find_data_dir() -> Path:
    """Locate the attached competition dataset under /kaggle/input."""
    hits = list(Path("/kaggle/input").rglob("input_2023_w01.csv"))
    assert hits, "Attach the NFL Big Data Bowl 2026 dataset as notebook input"
    return hits[0].parent


DATA_DIR = find_data_dir()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True          # speed on fixed shapes
print(f"torch={torch.__version__} device={DEVICE} "
      f"gpu={torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else '-'}")


@dataclass
class Config:
    # data / split (official-style week split)
    train_weeks: List[int] = field(default_factory=lambda: list(range(1, 16)))
    val_weeks: List[int] = field(default_factory=lambda: [16, 17])
    test_weeks: List[int] = field(default_factory=lambda: [18])
    max_input_frames: int = 64
    max_output_frames: int = 48
    min_output_frames: int = 5
    # model
    d_model: int = 128
    n_heads: int = 8
    n_temporal_blocks: int = 4
    n_spatial_layers_pre: int = 2
    n_refine_blocks: int = 2
    rnn_hidden: int = 96
    dropout: float = 0.1
    dist_bias_buckets: int = 16
    dist_bias_max: float = 40.0
    rope_base: float = 10000.0
    # training
    seed: int = 42
    epochs: int = 40
    batch_size: int = 32
    lr_muon: float = 0.02
    lr_adamw: float = 3e-4
    weight_decay: float = 0.01
    warmup_steps: int = 500
    huber_delta: float = 0.35
    ema_decay: float = 0.995
    grad_clip: float = 1.0
    patience: int = 6
    num_workers: int = 2          # Kaggle gives few CPUs
    amp: bool = True              # Disable AMP by default for stability of float32
    use_muon: bool = False        # By default, AdamW (stable)
    # statistics
    n_bootstrap: int = 1000
    bootstrap_block: int = 8
    exp_name: str = "main"

    def resolve_device(self) -> torch.device:
        return DEVICE


def exp_dirs(exp_name: str) -> Tuple[Path, Path]:
    out, fig = OUT_ROOT / exp_name, FIG_ROOT / exp_name
    out.mkdir(parents=True, exist_ok=True); fig.mkdir(parents=True, exist_ok=True)
    return out, fig


def ckpt_path(exp_name: str) -> Path:
    return CKPT_ROOT / f"{exp_name}_best.pt"


ROLE_LIST = ["Defensive Coverage", "Targeted Receiver", "Passer", "Other Route Runner"]
BASE = Config()   # base config for data building (seed/exp overridden per run)

## 2. Data Loading and Preprocessing (with cache)

In [ ]:
def _norm_angle(deg: np.ndarray) -> np.ndarray:
    return (deg + 180.0) % 360.0 - 180.0


def load_week(week: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    inp = pd.read_csv(DATA_DIR / f"input_2023_w{week:02d}.csv")
    out = pd.read_csv(DATA_DIR / f"output_2023_w{week:02d}.csv")
    return inp, out


def build_plays(inp: pd.DataFrame, out: pd.DataFrame) -> pd.DataFrame:
    inp = inp.sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
    out = out.sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
    g_out = out.groupby(["game_id", "play_id"], sort=False)
    rows = []
    for (gid, pid), gi in inp.groupby(["game_id", "play_id"], sort=False):
        try:
            go = g_out.get_group((gid, pid))
        except KeyError:
            continue
        first = gi.iloc[0]
        rows.append(dict(game_id=gid, play_id=pid, inp=gi, out=go,
                         ball_land_x=float(first.ball_land_x),
                         ball_land_y=float(first.ball_land_y)))
    return pd.DataFrame(rows)


def role_codes(inp: pd.DataFrame, players: np.ndarray) -> np.ndarray:
    mapping = inp.drop_duplicates("nfl_id").set_index("nfl_id").player_role
    return np.array([ROLE_LIST.index(mapping[p]) if mapping[p] in ROLE_LIST else 3
                     for p in players], np.int64)


def prepare_play(inp: pd.DataFrame, out: pd.DataFrame, cfg: Config):
    """One play -> float arrays. Offense always attacks +x (flip if left)."""
    players = inp.nfl_id.unique(); P = len(players)
    pidx = {p: i for i, p in enumerate(players)}
    flip = inp.play_direction.iloc[0] == "left"
    ang_shift = 180.0 if flip else 0.0

    S = min(int(inp.frame_id.max()), cfg.max_input_frames)
    inp = inp[inp.frame_id <= S]
    X = np.zeros((P, S, 2), np.float32); SA = np.zeros((P, S), np.float32)
    AA = np.zeros((P, S), np.float32); OO = np.zeros((P, S), np.float32)
    DD = np.zeros((P, S), np.float32); to_pred = np.zeros(P, np.float32)
    side = np.zeros(P, np.float32); is_target = np.zeros(P, np.float32)
    is_passer = np.zeros(P, np.float32)

    for nfl_id, gi in inp.groupby("nfl_id", sort=False):
        i = pidx[nfl_id]
        f = np.clip(gi.frame_id.to_numpy() - 1, 0, S - 1)
        x = gi.x.to_numpy(np.float32); y = gi.y.to_numpy(np.float32)
        if flip:
            x, y = 120.0 - x, 53.3 - y
        X[i, f, 0], X[i, f, 1] = x, y
        SA[i, f] = gi.s.to_numpy(np.float32); AA[i, f] = gi.a.to_numpy(np.float32)
        OO[i, f] = _norm_angle(gi.o.to_numpy(np.float32) + ang_shift)
        DD[i, f] = _norm_angle(gi.dir.to_numpy(np.float32) + ang_shift)
        to_pred[i] = float(gi.player_to_predict.iloc[0])
        side[i] = float(gi.player_side.iloc[0] == "Offense")
        is_target[i] = float(gi.player_role.iloc[0] == "Targeted Receiver")
        is_passer[i] = float(gi.player_role.iloc[0] == "Passer")

    dX = np.zeros_like(X); dX[:, 1:] = X[:, 1:] - X[:, :-1]
    t = np.arange(S, dtype=np.float32)
    time_decay = np.exp(-(S - 1 - t) / 10.0)[:, None]

    last_xy = X[:, -1].copy()
    land = np.array([inp.ball_land_x.iloc[0], inp.ball_land_y.iloc[0]], np.float32)
    if flip:
        land = np.array([120.0 - land[0], 53.3 - land[1]], np.float32)
    dist_ball = np.linalg.norm(last_xy - land[None], axis=-1)
    tgt = last_xy[is_target.astype(bool)][0] if is_target.any() else land
    dist_tgt = np.linalg.norm(last_xy - tgt[None], axis=-1)
    static = np.stack([last_xy[:, 0] / 120.0, last_xy[:, 1] / 53.3,
                       dist_ball / 60.0, dist_tgt / 60.0,
                       side, is_target, is_passer], axis=-1).astype(np.float32)

    O = min(int(out.frame_id.max()), cfg.max_output_frames)
    if O < cfg.min_output_frames:
        return None
    Y = np.zeros((P, O, 2), np.float32); valid = np.zeros((P, O), np.float32)
    for nfl_id, go in out.groupby("nfl_id", sort=False):
        i = pidx[nfl_id]
        f = go.frame_id.to_numpy() - 1
        m = f < O
        x = go.x.to_numpy(np.float32)[m]; y = go.y.to_numpy(np.float32)[m]
        if flip:
            x, y = 120.0 - x, 53.3 - y
        Y[i, f[m], 0], Y[i, f[m], 1] = x, y
        valid[i, f[m]] = 1.0

    prev = np.concatenate([last_xy[:, None, :], Y[:, :-1]], axis=1)
    dY = (Y - prev).astype(np.float32)
    return dict(dX=dX, X=X, s=SA, a=AA, o=OO, dir=DD, time_decay=time_decay,
                static=static, last_xy=last_xy.astype(np.float32), land=land,
                dY=dY, valid=valid, to_pred=to_pred, roles=role_codes(inp, players),
                S=np.int64(S), O=np.int64(O), P=np.int64(P))


def load_all_plays(cfg: Config) -> pd.DataFrame:
    weeks = sorted(set(cfg.train_weeks) | set(cfg.val_weeks) | set(cfg.test_weeks))
    frames = []
    for w in weeks:
        inp, out = load_week(w)
        df = build_plays(inp, out); df["week"] = w
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


# ---- build once, cache to /kaggle/working (reruns are free) ----------------
t0 = time.time()
if CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as f:
        PREP_CACHE = pickle.load(f)
    print(f"loaded prep cache: {len(PREP_CACHE)} plays")
else:
    PREP_CACHE = {}
PLAYS = load_all_plays(BASE)
print(f"raw plays: {len(PLAYS)}; prep {time.time()-t0:.1f}s so far")

## 3. Dataset, Colate, Loaders

In [ ]:
class NFLDataset(Dataset):
    def __init__(self, plays: pd.DataFrame, cfg: Config, weeks: List[int], cache: Dict):
        self.items = []
        for w in weeks:
            for _, row in plays[plays.week == w].iterrows():
                key = (w, row.game_id, row.play_id)
                if key not in cache:
                    cache[key] = prepare_play(row.inp, row.out, cfg)
                prep = cache[key]
                if prep is not None:
                    self.items.append(prep)

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        return {k: torch.from_numpy(np.asarray(v)) for k, v in self.items[idx].items()}


def collate_plays(batch):
    B = len(batch)
    P = max(int(b["P"]) for b in batch); S = max(int(b["S"]) for b in batch)
    O = max(int(b["O"]) for b in batch)

    def pad(t, shape):
        out = torch.zeros(shape, dtype=torch.float32)
        out[tuple(slice(0, d) for d in t.shape)] = t.float()
        return out

    out = {
        "dX": torch.stack([pad(b["dX"], (P, S, 2)) for b in batch]),
        "X": torch.stack([pad(b["X"], (P, S, 2)) for b in batch]),
        "s": torch.stack([pad(b["s"], (P, S)) for b in batch]),
        "a": torch.stack([pad(b["a"], (P, S)) for b in batch]),
        "o": torch.stack([pad(b["o"], (P, S)) for b in batch]),
        "dir": torch.stack([pad(b["dir"], (P, S)) for b in batch]),
        "time_decay": torch.stack([pad(b["time_decay"], (S, 1)) for b in batch]),
        "static": torch.stack([pad(b["static"], (P, 7)) for b in batch]),
        "last_xy": torch.stack([pad(b["last_xy"], (P, 2)) for b in batch]),
        "land": torch.stack([b["land"].float() for b in batch]),
        "dY": torch.stack([pad(b["dY"], (P, O, 2)) for b in batch]),
        "valid": torch.stack([pad(b["valid"], (P, O)) for b in batch]),
        "to_pred": torch.stack([pad(b["to_pred"], (P,)) for b in batch]),
        "roles": torch.stack([pad(b["roles"], (P,)) for b in batch]).long(),
        "S": torch.tensor([int(b["S"]) for b in batch], dtype=torch.long),
        "O": torch.tensor([int(b["O"]) for b in batch], dtype=torch.long),
    }
    p_mask, s_mask, o_mask = torch.zeros(B, P), torch.zeros(B, P, S), torch.zeros(B, P, O)
    for i, b in enumerate(batch):
        Pi, Si, Oi = int(b["P"]), int(b["S"]), int(b["O"])
        p_mask[i, :Pi] = 1.0; s_mask[i, :Pi, :Si] = 1.0; o_mask[i, :Pi, :Oi] = 1.0
    out.update(p_mask=p_mask, s_mask=s_mask, o_mask=o_mask)
    return out


def make_loader(ds, cfg, shuffle):
    g = torch.Generator(); g.manual_seed(cfg.seed)
    return DataLoader(ds, batch_size=cfg.batch_size, shuffle=shuffle,
                      num_workers=cfg.num_workers, collate_fn=collate_plays,
                      generator=g, persistent_workers=cfg.num_workers > 0,
                      pin_memory=DEVICE.type == "cuda")


t0 = time.time()
TRAIN_DS = NFLDataset(PLAYS, BASE, BASE.train_weeks, PREP_CACHE)
VAL_DS = NFLDataset(PLAYS, BASE, BASE.val_weeks, PREP_CACHE)
TEST_DS = NFLDataset(PLAYS, BASE, BASE.test_weeks, PREP_CACHE)
with open(CACHE_PATH, "wb") as f:          # persist for future reruns
    pickle.dump(PREP_CACHE, f)
print(f"train={len(TRAIN_DS)} val={len(VAL_DS)} test={len(TEST_DS)} "
      f"prep_total={time.time()-t0:.1f}s")

## 4. Architecture

In [ ]:
def rope_angles(positions, dim, base):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2, device=positions.device,
                                       dtype=torch.float32) / dim))
    return positions[..., None].float() * inv[None, :]


def apply_rope(x, angles):
    d = x.shape[-1]
    x1, x2 = x[..., 0:d:2], x[..., 1:d:2]
    cos, sin = torch.cos(angles), torch.sin(angles)
    out = torch.empty_like(x)
    out[..., 0:d:2] = x1 * cos - x2 * sin
    out[..., 1:d:2] = x1 * sin + x2 * cos
    return out

def _mask_zero(x: torch.Tensor, key_pad: torch.Tensor) -> torch.Tensor:
    """
    NaN-safe padding zeroing: REPLACE (not multiply!) padded token embeddings
    with exact zeros. Multiplication would keep NaN (NaN * 0 = NaN).
    """
    return torch.where(key_pad.unsqueeze(-1), torch.zeros_like(x), x)
    


class DistanceBias(nn.Module):
    def __init__(self, n_buckets, max_dist, n_heads):
        super().__init__()
        self.n_buckets, self.max_dist = n_buckets, max_dist
        self.emb = nn.Embedding(n_buckets, n_heads); nn.init.zeros_(self.emb.weight)

    def forward(self, dist):
        b = (dist / self.max_dist * (self.n_buckets - 1)).clamp(0, self.n_buckets - 1).long()
        return self.emb(b).permute(0, 3, 1, 2)


class MaskedAttention(nn.Module):
    """Manual MHA with key padding mask, 4-D additive bias and dead-row fix."""

    def __init__(self, dim, n_heads, dropout):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads, self.dh = n_heads, dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_pad=None, attn_bias=None):
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        shape = (B, L, self.n_heads, self.dh)
        q, k, v = [t.view(*shape).transpose(1, 2) for t in (q, k, v)]
        scores = q @ k.transpose(-1, -2) / math.sqrt(self.dh)
        if attn_bias is not None:
            scores = scores + attn_bias
        if key_pad is not None:
            scores = scores.masked_fill(key_pad[:, None, None, :], -torch.inf)
            # FIX: rows where ALL keys are masked (padded player / padded frame)
            # would give softmax([-inf,...]) = NaN -> make them uniform instead.
            dead = key_pad.all(dim=-1)                      # (B,)
            if dead.any():
                scores = scores.masked_fill(dead[:, None, None, None], 0.0)
        attn = self.drop(torch.softmax(scores, dim=-1))
        return self.out((attn @ v).transpose(1, 2).reshape(B, L, D))


class FeedForward(nn.Module):
    def __init__(self, dim, mult, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim * mult), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(dim * mult, dim))

    def forward(self, x): return self.net(x)


class TransformerEncoderLayer(nn.Module):
    """Pre-LN Transformer layer; padded tokens are REPLACED with zeros."""

    def __init__(self, dim, n_heads, dropout, ffn_mult=4):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attn = MaskedAttention(dim, n_heads, dropout)
        self.ff = FeedForward(dim, ffn_mult, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_pad=None, attn_bias=None):
        x = x + self.drop(self.attn(self.ln1(x), key_pad, attn_bias))
        x = x + self.drop(self.ff(self.ln2(x)))
        if key_pad is not None:
            x = _mask_zero(x, key_pad)
        return x


class SqueezeFormerBlock(nn.Module):
    """FFN-Attention-Conv-FFN block for the temporal axis; NaN-safe padding."""

    def __init__(self, dim, n_heads, dropout):
        super().__init__()
        self.ln = nn.ModuleList([nn.LayerNorm(dim) for _ in range(4)])
        self.ff1 = FeedForward(dim, 2, dropout)
        self.attn = MaskedAttention(dim, n_heads, dropout)
        self.conv = nn.Sequential(nn.Conv1d(dim, dim, 3, padding=1), nn.GELU(),
                                  nn.Conv1d(dim, dim, 3, padding=1, groups=dim), nn.GELU())
        self.ff2 = FeedForward(dim, 4, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_pad=None):
        x = x + 0.5 * self.drop(self.ff1(self.ln[0](x)))
        x = x + self.drop(self.attn(self.ln[1](x), key_pad))
        h = self.ln[2](x)
        if key_pad is not None:
            h = _mask_zero(h, key_pad)          # conv must not see NaN/garbage
        x = x + self.drop(self.conv(h.transpose(1, 2)).transpose(1, 2))
        x = x + 0.5 * self.drop(self.ff2(self.ln[3](x)))
        if key_pad is not None:
            x = _mask_zero(x, key_pad)
        return x


class SpatioTemporalBlockV1(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.temporal = SqueezeFormerBlock(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.spatial = TransformerEncoderLayer(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.dist_bias = DistanceBias(cfg.dist_bias_buckets, cfg.dist_bias_max, cfg.n_heads)

    def forward(self, x, s_mask, dist):
        B, P, S, D = x.shape
        xt = self.temporal(x.reshape(B * P, S, D), key_pad=(s_mask.reshape(B * P, S) < 0.5))
        xs = xt.reshape(B, P, S, D).permute(0, 2, 1, 3).reshape(B * S, P, D)
        pad_s = s_mask.permute(0, 2, 1).reshape(B * S, P) < 0.5
        bias = self.dist_bias(dist).unsqueeze(1).expand(B, S, -1, -1, -1).reshape(B * S, -1, P, P)
        xs = self.spatial(xs, key_pad=pad_s, attn_bias=bias)
        return xs.reshape(B, S, P, D).permute(0, 2, 1, 3)


class SpatioTemporalBlockV2(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.temporal = TransformerEncoderLayer(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.spatial = TransformerEncoderLayer(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.dist_bias = DistanceBias(cfg.dist_bias_buckets, cfg.dist_bias_max, cfg.n_heads)

    def forward(self, x, o_mask, dist):
        B, P, O, D = x.shape
        xt = self.temporal(x.reshape(B * P, O, D), key_pad=(o_mask.reshape(B * P, O) < 0.5))
        xs = xt.reshape(B, P, O, D).permute(0, 2, 1, 3).reshape(B * O, P, D)
        pad_s = o_mask.permute(0, 2, 1).reshape(B * O, P) < 0.5
        bias = self.dist_bias(dist).unsqueeze(1).expand(B, O, -1, -1, -1).reshape(B * O, -1, P, P)
        xs = self.spatial(xs, key_pad=pad_s, attn_bias=bias)
        return xs.reshape(B, O, P, D).permute(0, 2, 1, 3)


class TrajectoryModel(nn.Module):
    STATIC_DIM, IN_FEAT_DIM, OUT_FEAT_DIM = 7, 9, 4

    def __init__(self, cfg):
        super().__init__()
        d = cfg.d_model
        self.temporal_proj = nn.Sequential(nn.Linear(self.IN_FEAT_DIM, d), nn.GELU(), nn.Linear(d, d))
        self.v1 = nn.ModuleList([SpatioTemporalBlockV1(cfg) for _ in range(cfg.n_temporal_blocks)])
        self.static_proj = nn.Linear(d + self.STATIC_DIM, d)
        self.spatial_pre = nn.ModuleList([TransformerEncoderLayer(d, cfg.n_heads, cfg.dropout)
                                          for _ in range(cfg.n_spatial_layers_pre)])
        self.dist_bias_pre = DistanceBias(cfg.dist_bias_buckets, cfg.dist_bias_max, cfg.n_heads)
        self.out_feat_proj = nn.Linear(d + self.OUT_FEAT_DIM, d)
        self.bilstm = nn.LSTM(d, cfg.rnn_hidden, 2, batch_first=True, bidirectional=True, dropout=cfg.dropout)
        self.bigru = nn.GRU(d, cfg.rnn_hidden, 2, batch_first=True, bidirectional=True, dropout=cfg.dropout)
        self.mix_proj = nn.Linear(4 * cfg.rnn_hidden + d, d)
        self.v2 = nn.ModuleList([SpatioTemporalBlockV2(cfg) for _ in range(cfg.n_refine_blocks)])
        self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Linear(d, 2))
        self.cfg = cfg

    def forward(self, batch):
        cfg = self.cfg
        B, P, S, _ = batch["X"].shape
        O = int(batch["dY"].shape[2])
        feats = torch.cat([
            batch["dX"], batch["s"][..., None] / 10.0, batch["a"][..., None] / 20.0,
            torch.sin(torch.deg2rad(batch["o"]))[..., None],
            torch.cos(torch.deg2rad(batch["o"]))[..., None],
            torch.sin(torch.deg2rad(batch["dir"]))[..., None],
            torch.cos(torch.deg2rad(batch["dir"]))[..., None],
            batch["time_decay"].unsqueeze(1).expand(B, P, S, 1),
        ], dim=-1)
        h = self.temporal_proj(feats)
        dist = torch.cdist(batch["last_xy"], batch["last_xy"])
        for blk in self.v1:
            h = blk(h, batch["s_mask"], dist)
        idx = (batch["S"] - 1).clamp(min=0)[:, None, None, None].expand(B, P, 1, cfg.d_model)
        h_last = torch.gather(h, 2, idx).squeeze(2)
        h = self.static_proj(torch.cat([h_last, batch["static"]], dim=-1))
        bias = self.dist_bias_pre(dist)
        for lyr in self.spatial_pre:
            h = lyr(h, key_pad=(batch["p_mask"] < 0.5), attn_bias=bias)

        horizon = torch.arange(O, device=h.device)
        angles = rope_angles(horizon, cfg.d_model, cfg.rope_base)
        h = apply_rope(h.unsqueeze(2).expand(B, P, O, cfg.d_model),
                       angles[None, None].expand(B, P, O, cfg.d_model // 2))
        # --- FIX: every horizon feature is expanded over the player axis ----
        t_sec = (horizon + 1).float() / 10.0                              # (O,)
        phase = torch.deg2rad(horizon * 36.0)                             # (O,)
        o_len = batch["O"][:, None, None, None].float().clamp(min=1.0)    # (B,1,1,1)
        prog = (horizon + 1).float()[None, None, :, None] / o_len         # (B,1,O,1)
        out_feats = torch.cat([
            t_sec[None, None, :, None].expand(B, P, O, 1),
            prog.expand(B, P, O, 1),
            torch.sin(phase)[None, None, :, None].expand(B, P, O, 1),
            torch.cos(phase)[None, None, :, None].expand(B, P, O, 1),
        ], dim=-1)                                                        # (B,P,O,4)
        h = self.out_feat_proj(torch.cat([h, out_feats], dim=-1))

        ht = h.reshape(B * P, O, cfg.d_model)
        lengths = batch["o_mask"].reshape(B * P, O).sum(-1).clamp(min=1).cpu().long()
        packed = nn.utils.rnn.pack_padded_sequence(ht, lengths, batch_first=True, enforce_sorted=False)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(self.bilstm(packed)[0], batch_first=True)
        gru_out, _ = nn.utils.rnn.pad_packed_sequence(self.bigru(packed)[0], batch_first=True)
        h = self.mix_proj(torch.cat([lstm_out, gru_out, ht], dim=-1)).reshape(B, P, O, cfg.d_model)
        for blk in self.v2:
            h = blk(h, batch["o_mask"], dist)
        return self.head(h)


# shape self-test
_cfg = Config(); _B, _P, _S, _O = 4, 22, 30, 25
_tb = dict(dX=torch.randn(_B,_P,_S,2), X=torch.randn(_B,_P,_S,2), s=torch.randn(_B,_P,_S),
           a=torch.randn(_B,_P,_S), o=torch.randn(_B,_P,_S), dir=torch.randn(_B,_P,_S),
           time_decay=torch.randn(_B,_S,1), static=torch.randn(_B,_P,7),
           last_xy=torch.randn(_B,_P,2), land=torch.randn(_B,2), dY=torch.randn(_B,_P,_O,2),
           valid=torch.ones(_B,_P,_O), to_pred=torch.ones(_B,_P),
           roles=torch.zeros(_B,_P,dtype=torch.long), p_mask=torch.ones(_B,_P),
           s_mask=torch.ones(_B,_P,_S), o_mask=torch.ones(_B,_P,_O),
           S=torch.full((_B,),_S,dtype=torch.long), O=torch.full((_B,),_O,dtype=torch.long))
with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
    _out = TrajectoryModel(_cfg)(_tb)
assert _out.shape == (_B, _P, _O, 2)
print("forward ok:", tuple(_out.shape))

## 5. Code (Optimizers, EMA, Losses, Metrics, Bootstrap)

In [ ]:
def _zeropower_via_newton_schulz(g, steps=5):
    a, b, c = (3.4445, -4.7750, 2.0315)
    transposed = g.shape[-2] > g.shape[-1]
    x = g.mT if transposed else g
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    for _ in range(steps):
        A = x @ x.mT
        x = a * x + b * (A @ x) + c * ((A @ A) @ x)
    return x.mT if transposed else x


class Muon(torch.optim.Optimizer):
    def __init__(self, params, lr=0.02, weight_decay=0.01, momentum=0.95):
        super().__init__(params, dict(lr=lr, weight_decay=weight_decay, momentum=momentum))

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            lr, wd, mu = group["lr"], group["weight_decay"], group["momentum"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                st = self.state[p]
                if "momentum" not in st:
                    st["momentum"] = torch.zeros_like(p.grad)
                st["momentum"].lerp_(p.grad, 1 - mu)
                update = _zeropower_via_newton_schulz(st["momentum"])
                scale = 0.2 * math.sqrt(max(p.shape[-2], p.shape[-1]))
                p.mul_(1 - lr * wd).add_(update, alpha=-lr * scale)


def build_optimizers(model, cfg):
    """Builds AdamW for everything by default; optionally splits to Muon for 2D weights."""
    muon_p, adamw_p = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        # Use Muon ONLY if explicitly enabled AND it's a 2D hidden weight
        if cfg.use_muon and p.ndim >= 2 and not any(k in name for k in ("head", "emb", "ln")):
            muon_p.append(p)
        else:
            adamw_p.append(p)
            
    opts = []
    if muon_p:
        opts.append(Muon(muon_p, lr=cfg.lr_muon, weight_decay=cfg.weight_decay))
    if adamw_p:
        opts.append(torch.optim.AdamW(adamw_p, lr=cfg.lr_adamw, weight_decay=cfg.weight_decay))
    return opts


class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.is_floating_point():
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def state_dict(self): return self.shadow
    def copy_to(self, model): model.load_state_dict(self.shadow)


@contextmanager
def ema_weights(model, ema):
    live = {k: v.detach().clone() for k, v in model.state_dict().items()}
    ema.copy_to(model)
    try:
        yield
    finally:
        model.load_state_dict(live)


def huber_delta_loss(pred, target, mask, delta):
    loss = F.huber_loss(pred, target, reduction="none", delta=delta).sum(-1)
    return (loss * mask).sum() / mask.sum().clamp(min=1.0)


def official_rmse(err_sq_sum, n_points):
    return float(math.sqrt(err_sq_sum / (2.0 * max(n_points, 1))))


def deltas_to_absolute(pred_delta: np.ndarray, anchor: np.ndarray) -> np.ndarray:
    """FIX: anchor (..., 2) -> (..., 1, 2); broadcasts with (..., O, 2)."""
    return anchor[..., None, :] + np.cumsum(pred_delta, axis=-2)


def bootstrap_rmse_ci(plays_err, rng, n_boot, block):
    point = official_rmse(float(sum(e.sum() for e in plays_err)),
                          int(sum(e.size for e in plays_err)))
    n = len(plays_err)
    if n == 0:
        return point, point, point
    stats = []
    for _ in range(n_boot):
        sampled, taken = [], 0
        while taken < n:
            start = int(rng.integers(0, n))
            sampled.extend(plays_err[start:start + block]); taken += block
        stats.append(official_rmse(float(sum(e.sum() for e in sampled)),
                                   int(sum(e.size for e in sampled))))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return point, float(lo), float(hi)

## 6. Train / Evaluate / Baseline (AMP + tqdm.notebook)

In [ ]:
if hasattr(torch.amp, "GradScaler"):
    def make_scaler(en): return torch.amp.GradScaler("cuda", enabled=en)
else:
    def make_scaler(en): return torch.cuda.amp.GradScaler(enabled=en)


def make_autocast(en):
    return torch.autocast("cuda", dtype=torch.float16, enabled=en)


def to_device(batch, device):
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v)
            for k, v in batch.items()}


def lr_lambda_factory(total_steps, warmup):
    def fn(step):
        if step < warmup:
            return step / max(1, warmup)
        prog = (step - warmup) / max(1, total_steps - warmup)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * min(prog, 1.0))))
    return fn


@torch.no_grad()
def evaluate_loss_rmse(model, loader, cfg, device):
    model.eval()
    tot_loss = tot_sq = tot_n = 0.0
    ac = make_autocast(cfg.amp and device.type == "cuda")
    for batch in loader:
        batch = to_device(batch, device)
        with ac:
            pred = model(batch)
            mask = batch["valid"] * batch["to_pred"][..., None]
            # FIX 1: Cast to float32 to prevent fp16 overflow in loss calculation
            loss = huber_delta_loss(pred.float(), batch["dY"], mask, cfg.huber_delta)
        tot_loss += loss.item() * mask.sum().item()
        pred_abs = deltas_to_absolute(pred.float().cpu().numpy(), batch["last_xy"].cpu().numpy())
        true_abs = deltas_to_absolute(batch["dY"].cpu().numpy(), batch["last_xy"].cpu().numpy())
        m = mask.cpu().numpy() > 0
        tot_sq += float(((pred_abs - true_abs) ** 2).sum(-1)[m].sum()); tot_n += int(m.sum())
    return tot_loss / max(1, tot_n), official_rmse(tot_sq, tot_n)


def train(cfg, train_ds, val_ds):
    set_seed(cfg.seed)
    device = cfg.resolve_device()
    out_dir, fig_dir = exp_dirs(cfg.exp_name)
    train_loader = make_loader(train_ds, cfg, True)
    val_loader = make_loader(val_ds, cfg, False)

    model = TrajectoryModel(cfg).to(device)
    print(f"[info] exp={cfg.exp_name} params="
          f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    opts = build_optimizers(model, cfg)  # Теперь это список [AdamW] или [Muon, AdamW]
    total_steps = max(1, len(train_loader) * cfg.epochs)
    lam = lr_lambda_factory(total_steps, cfg.warmup_steps)
    scheds = [LambdaLR(opt, lam) for opt in opts]
    
    ema = EMA(model, cfg.ema_decay)
    use_amp = cfg.amp and device.type == "cuda"
    scaler = make_scaler(use_amp)
    ac = make_autocast(use_amp)

    history = dict(train_loss=[], val_loss=[], val_rmse=[], best_epoch=-1, best_val_rmse=math.inf)
    bad = 0
    for epoch in range(cfg.epochs):
        model.train()
        loss_sum = mask_sum = 0.0
        pbar = tqdm(train_loader, desc=f"train ep{epoch:02d}", leave=False)
        for batch in pbar:
            batch = to_device(batch, device)
            for opt in opts:
                opt.zero_grad(set_to_none=True)
                
            with ac:
                pred = model(batch)
                mask = batch["valid"] * batch["to_pred"][..., None]
                loss = huber_delta_loss(pred.float(), batch["dY"], mask, cfg.huber_delta)
                
            scaler.scale(loss).backward()
            
            # Unscale, clip, step for ALL optimizers
            for opt in opts:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], cfg.grad_clip)
            for opt in opts:
                scaler.step(opt)
            scaler.update()
            
            for s in scheds:
                s.step()
            ema.update(model)
            
            n = mask.sum().item(); loss_sum += loss.item() * n; mask_sum += n
            pbar.set_postfix(loss=loss.item(), lr=opts[-1].param_groups[0]["lr"])
            
        current_loss = loss_sum / max(1.0, mask_sum)
        if math.isnan(current_loss):
            print(f"[WARNING] Training loss became NaN at epoch {epoch}!")
            
        history["train_loss"].append(current_loss)
        with ema_weights(model, ema):
            val_loss, val_rmse = evaluate_loss_rmse(model, val_loader, cfg, device)
        history["val_loss"].append(val_loss); history["val_rmse"].append(val_rmse)
        print(f"epoch {epoch:02d} | train_loss={current_loss:.4f} "
              f"val_loss={val_loss:.4f} val_rmse={val_rmse:.4f}")
              
        if val_rmse < history["best_val_rmse"] - 1e-4:
            history.update(best_val_rmse=val_rmse, best_epoch=epoch); bad = 0
            torch.save(dict(model=model.state_dict(), ema=ema.state_dict(),
                            cfg=asdict(cfg), epoch=epoch, val_rmse=val_rmse),
                       ckpt_path(cfg.exp_name))
        else:
            bad += 1
            if math.isnan(val_rmse) or math.isinf(val_rmse) or history["best_epoch"] == -1:
                torch.save(dict(model=model.state_dict(), ema=ema.state_dict(),
                                cfg=asdict(cfg), epoch=epoch, val_rmse=val_rmse),
                           ckpt_path(cfg.exp_name))
            if bad >= cfg.patience:
                print(f"[info] early stopping at epoch {epoch}"); break
                
    with open(out_dir / "history.json", "w") as f:
        json.dump(history, f, indent=2)
    return history


@torch.no_grad()
def collect_test_statistics(model, loader, cfg, device, n_samples=4):
    model.eval()
    plays_err, hor_h, hor_e, radial = [], [], [], []
    role_rmse, samples = {}, []
    ac = make_autocast(cfg.amp and device.type == "cuda")
    for batch in tqdm(loader, desc="test eval", leave=False):
        batch = to_device(batch, device)
        with ac:
            pred = model(batch).float().cpu().numpy()
        pred_abs = deltas_to_absolute(pred, batch["last_xy"].cpu().numpy())
        true_abs = deltas_to_absolute(batch["dY"].cpu().numpy(), batch["last_xy"].cpu().numpy())
        mask = (batch["valid"].cpu().numpy() > 0) & (batch["to_pred"].cpu().numpy() > 0)[..., None]
        sq = ((pred_abs - true_abs) ** 2).sum(-1)
        radial.append(np.sqrt(sq[mask]))
        for b in range(sq.shape[0]):
            Pi, Oi = int(batch["p_mask"][b].sum()), int(batch["o_mask"][b, 0].sum())
            m, sq_b = mask[b, :Pi, :Oi], sq[b, :Pi, :Oi]
            plays_err.append(sq_b[m])
            hh = np.arange(Oi)[None, :].repeat(Pi, 0)
            hor_h.append(hh[m]); hor_e.append(sq_b[m])
            # Added .cpu() for GPU tensors before .numpy()
            roles = batch["roles"][b, :Pi].cpu().numpy()
            for p in range(Pi):
                if m[p].any():
                    role_rmse.setdefault(int(roles[p]), []).append(
                        float(np.sqrt(sq_b[p][m[p]].mean() / 2)))
            if len(samples) < n_samples:
                samples.append(dict(play=f"{int(batch['S'][b])}in/{Oi}out",
                                    land=batch["land"][b].cpu().numpy(), # FIX
                                    players=[(ROLE_LIST[int(roles[p])], true_abs[b, p, :Oi],
                                              pred_abs[b, p, :Oi], 
                                              batch["last_xy"][b, p].cpu().numpy()) # FIX
                                             for p in range(Pi) if m[p].any()]))
    return dict(plays_err=plays_err, horizon=(np.concatenate(hor_h), np.concatenate(hor_e)),
                radial=np.concatenate(radial), role_rmse=role_rmse, samples=samples)


@torch.no_grad()
def constant_velocity_baseline(loader, device): # FIX: added device
    errs, hors = [], []
    for batch in tqdm(loader, desc="baseline cv", leave=False):
        batch = to_device(batch, device) # FIX
        O = int(batch["dY"].shape[2])
        pred = batch["dX"][:, :, -1:].expand(-1, -1, O, -1)
        # Added .cpu()
        pred_abs = deltas_to_absolute(pred.cpu().numpy(), batch["last_xy"].cpu().numpy())
        true_abs = deltas_to_absolute(batch["dY"].cpu().numpy(), batch["last_xy"].cpu().numpy())
        mask = (batch["valid"].cpu().numpy() > 0) & (batch["to_pred"].cpu().numpy() > 0)[..., None]
        sq = ((pred_abs - true_abs) ** 2).sum(-1)
        for b in range(sq.shape[0]):
            Pi, Oi = int(batch["p_mask"][b].sum()), int(batch["o_mask"][b, 0].sum())
            m = mask[b, :Pi, :Oi]
            errs.append(sq[b, :Pi, :Oi][m])
            hors.append(np.arange(Oi)[None, :].repeat(Pi, 0)[m])
    return errs, hors

### Sanity check

In [ ]:
_loader = make_loader(TRAIN_DS, BASE, shuffle=False)
_batch = to_device(next(iter(_loader)), DEVICE)
_m = TrajectoryModel(BASE).to(DEVICE)
_out = _m(_batch)
_loss = huber_delta_loss(_out.float(), _batch["dY"],
                         _batch["valid"] * _batch["to_pred"][..., None], BASE.huber_delta)
_loss.backward()
assert torch.isfinite(_out).all(), "NaN in forward output!"
assert torch.isfinite(_loss).all(), "NaN in loss!"
assert all(p.grad is None or torch.isfinite(p.grad).all()
           for p in _m.parameters()), "NaN in gradients!"
print("sanity ok: forward/loss/grads are finite")

## 7. Visualization

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.grid": True, "grid.alpha": 0.3})
ROLE_COLORS = {"Targeted Receiver": "#c44e52", "Defensive Coverage": "#4c72b0",
               "Passer": "#55a868", "Other Route Runner": "#999999"}


def plot_training_history(history, path):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history["train_loss"], label="train huber")
    axes[0].plot(history["val_loss"], label="val huber (EMA)")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
    axes[1].plot(history["val_rmse"], color="#c44e52")
    if history.get("best_epoch", -1) >= 0:
        axes[1].axvline(history["best_epoch"], ls=":", c="gray")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("RMSE")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def plot_rmse_comparison(results, path):
    names = list(results)
    point = np.array([results[n]["rmse"] for n in names])
    lo = np.array([results[n]["ci_lo"] for n in names])
    hi = np.array([results[n]["ci_hi"] for n in names])
    fig, ax = plt.subplots(figsize=(7, 4.2))
    xs = np.arange(len(names))
    ax.bar(xs, point, yerr=np.vstack([point - lo, hi - point]), capsize=6,
           color=["#4c72b0", "#dd8452"][:len(names)])
    ax.set_xticks(xs); ax.set_xticklabels(names, rotation=12)
    ax.set_ylabel("test RMSE (yards)")
    ax.set_title("Test-week comparison (95% CI, block bootstrap)")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def plot_rmse_vs_horizon(per_frame, path, bin_size=4):
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for name, (h, e) in per_frame.items():
        if h.size == 0:
            continue
        bins = np.arange(0, h.max() + bin_size, bin_size)
        idx = np.digitize(h, bins) - 1
        c, m = [], []
        for i in range(len(bins) - 1):
            sel = idx == i
            if sel.any():
                c.append(bins[i] + bin_size / 2); m.append(np.sqrt(e[sel].mean() / 2))
        ax.plot(c, m, marker="o", ms=4, label=name)
    ax.set_xlabel("future frame o"); ax.set_ylabel("RMSE(o) (yards)")
    ax.set_title("Error growth over horizon"); ax.legend()
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def plot_role_breakdown(role_stats, role_order, path):
    data, labels, colors = [], [], []
    for name, per_role in role_stats.items():
        for role in role_order:
            if role in per_role and per_role[role].size:
                data.append(per_role[role]); labels.append(f"{name}\n{role}")
                colors.append(ROLE_COLORS.get(role, "#999999"))
    if not data:
        return
    fig, ax = plt.subplots(figsize=(max(6, 1.6 * len(data)), 4.2))
    bp = ax.boxplot(data, labels=labels, showfliers=False, patch_artist=True)
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c); patch.set_alpha(0.55)
    ax.set_ylabel("per-player RMSE (yards)")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def plot_trajectories(samples, path):
    n = min(4, len(samples))
    if n == 0:
        return
    fig, axes = plt.subplots(1, n, figsize=(4.3 * n, 4.3), squeeze=False)
    for ax, s in zip(axes[0], samples[:n]):
        for role, true, pred, last in s["players"]:
            c = ROLE_COLORS.get(role, "#999999")
            ax.plot(true[:, 0], true[:, 1], "-", c=c, lw=1.4)
            ax.plot(pred[:, 0], pred[:, 1], "--", c=c, lw=1.1)
            ax.plot(last[0], last[1], "o", c=c, ms=4)
        ax.plot(s["land"][0], s["land"][1], "*", c="gold", ms=14, mec="k")
        ax.set_aspect("equal"); ax.set_title(s["play"])
    fig.suptitle("True (solid) vs predicted (dashed)")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def plot_error_qq(radial, path):
    qs = np.linspace(1, 99, 99)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(qs, np.percentile(radial, qs), marker=".", ms=3, ls="none")
    ax.set_xlabel("percentile"); ax.set_ylabel("radial error (yards)")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def plot_seed_summary(per_exp_rmse, path):
    names = list(per_exp_rmse); vals = np.array([per_exp_rmse[n] for n in names])
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.bar(np.arange(len(names)), vals, color="#4c72b0", alpha=0.8)
    if len(vals) > 1:
        ax.axhline(vals.mean(), c="#c44e52", ls="--",
                   label=f"mean {vals.mean():.4f} ± {vals.std(ddof=1):.4f}")
        ax.legend()
    ax.set_xticks(np.arange(len(names))); ax.set_xticklabels(names, rotation=12)
    ax.set_ylabel("test RMSE (yards)")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)

## 8. Evaluate + Run (1 seed) + Artifacts Wrapping

In [ ]:
def evaluate(cfg, test_ds):
    set_seed(cfg.seed)
    device = cfg.resolve_device()
    out_dir, fig_dir = exp_dirs(cfg.exp_name)
    loader = make_loader(test_ds, cfg, False)
    ckpt = torch.load(ckpt_path(cfg.exp_name), map_location="cpu", weights_only=False)
    model = TrajectoryModel(cfg); model.load_state_dict(ckpt["ema"]); model.to(device)

    stats = collect_test_statistics(model, loader, cfg, device)
    point, lo, hi = bootstrap_rmse_ci(stats["plays_err"], np.random.default_rng(cfg.seed),
                                      cfg.n_bootstrap, cfg.bootstrap_block)
    results = {"proposed (EMA)": dict(rmse=point, ci_lo=lo, ci_hi=hi)}
    
    # Pass device to baseline
    cv_err, cv_hor = constant_velocity_baseline(loader, device)
    p2, l2, h2 = bootstrap_rmse_ci(cv_err, np.random.default_rng(cfg.seed),
                                   cfg.n_bootstrap, cfg.bootstrap_block)
    results["constant-velocity"] = dict(rmse=p2, ci_lo=l2, ci_hi=h2)

    plot_rmse_comparison(results, fig_dir / "rmse_comparison.png")
    plot_rmse_vs_horizon({"proposed": stats["horizon"],
                          "constant-velocity": (np.concatenate(cv_hor), np.concatenate(cv_err))},
                         fig_dir / "rmse_vs_horizon.png")
    plot_role_breakdown({"proposed": {ROLE_LIST[r]: np.array(v)
                                      for r, v in stats["role_rmse"].items()}},
                        ROLE_LIST, fig_dir / "role_breakdown.png")
    plot_trajectories(stats["samples"], fig_dir / "trajectories.png")
    plot_error_qq(stats["radial"], fig_dir / "error_qq.png")

    report = dict(exp=cfg.exp_name, n_test_plays=len(test_ds), results=results,
                  role_rmse={"proposed": {ROLE_LIST[r]: float(np.mean(v))
                                          for r, v in stats["role_rmse"].items()}},
                  best_epoch=ckpt["epoch"])
    with open(out_dir / "test_results.json", "w") as f:
        json.dump(report, f, indent=2)
    print(json.dumps(report, indent=2))
    return report

SEEDS = [42]
reports = {}
for seed in SEEDS:
    # use_muon=False (по умолчанию) обеспечит стабильное обучение
    # amp=False (по умолчанию) защитит от переполнения fp16 на GPU
    cfg = Config(seed=seed, exp_name=f"seed_{seed}")
    history = train(cfg, TRAIN_DS, VAL_DS)
    plot_training_history(history, exp_dirs(cfg.exp_name)[1] / "training_history.png")
    reports[cfg.exp_name] = evaluate(cfg, TEST_DS)

summary = dict(exps=list(reports),
               rmse_per_seed=[reports[e]["results"]["proposed (EMA)"]["rmse"] for e in reports],
               mean=float(np.mean([reports[e]["results"]["proposed (EMA)"]["rmse"] for e in reports])),
               std=float(np.std([reports[e]["results"]["proposed (EMA)"]["rmse"] for e in reports], ddof=1)))
print(json.dumps(summary, indent=2))
plot_seed_summary({e: reports[e]["results"]["proposed (EMA)"]["rmse"] for e in reports},
                  FIG_ROOT / "seed_summary.png")
with open(OUT_ROOT / "seed_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

!cd /kaggle/working && zip -qr artifacts.zip checkpoints outputs figures
print("Download /kaggle/working/artifacts.zip (Output pane -> Download).")